# Plan C — Pretrained Dataset (D4RL / Minari) + Post-hoc Phase Labeling

## 배경
- Plan A (PhaseAwareWrapper + PPO): 학습 실패 (reward shaping의 한계)
- Plan B (Vanilla SAC + Hilbert): 작동하지만 17 episode + 단일 frequency만 확보
- **Plan C**: *학습 자체를 우회*. 이미 학습된 정책의 trajectory dataset을 다운로드.

## 핵심 아이디어
D4RL/Minari에 *expert 수준의 Ant trajectory가 수십만 step 패키징*되어 있음. 학습 시간 0, 다운로드 1분. 우리는 이 trajectory에 Plan B의 phase 추출 + quality 필터링을 적용.

## 데이터셋 비교
| Dataset | Source | Quality | Multi-freq 가능성 |
|---|---|---|---|
| `ant-random-v2` | random policy | 매우 낮음 | 높음 (그러나 보행 아님) |
| `ant-medium-v2` | 중간 학습 SAC | 중간 | 중간 |
| `ant-expert-v2` | 잘 학습된 SAC | 높음 | 낮음 (단일 frequency 가능성) |
| `ant-medium-replay-v2` | 학습 중 replay buffer | 다양 | **높음** ⭐ |
| `ant-medium-expert-v2` | medium + expert 합침 | 다양 | 중간~높음 |

**`ant-medium-replay-v2` 추천**: 학습 *과정 전체*의 trajectory라 *다양한 quality + 다양한 행동 패턴* → multi-frequency demo 확보 가능성 가장 높음.

## Framing
> *"우리는 D4RL의 ant-medium-replay-v2 dataset에서 trajectory를 가져와, Hilbert transform으로 phase quality를 측정한 뒤 phase-coherent한 subset을 자동 선택한다. 외부 학습 없이 다양한 보행 패턴의 demonstration을 확보한다."*


## Cell 1 — Setup

D4RL은 다소 무거운 dependency. Minari가 후속이라 더 가벼움. 둘 다 시도.

In [ ]:
!apt-get update -qq
!apt-get install -y libosmesa6-dev libgl1-mesa-glx libglfw3 patchelf --quiet
!pip install -q -r requirements.txt || pip install -q -r ../requirements.txt

## Cell 2 — Imports

⚠️ 런타임 재시작 직후 *가장 먼저* 실행.

In [ ]:
import os
os.environ['MUJOCO_GL'] = 'osmesa'
os.environ['PYOPENGL_PLATFORM'] = 'osmesa'

import gc
import time
import numpy as np
import gymnasium as gym
import imageio
import matplotlib.pyplot as plt
from IPython.display import Video, display

from scipy.signal import hilbert
from scipy.ndimage import gaussian_filter1d

import minari

print(f"Minari version: {minari.__version__}")

## Cell 3 — 사용 가능한 Ant dataset 탐색

Minari의 remote dataset 목록에서 ant 관련 모두 찾기.

In [ ]:
print("=== Minari에서 사용 가능한 Ant dataset ===")
remote = minari.list_remote_datasets()
ant_datasets = [name for name in remote.keys() if 'ant' in name.lower()]

if not ant_datasets:
    print("⚠ Ant dataset 없음. D4RL 직접 시도 필요.")
    print("\n전체 dataset (참고):")
    for name in list(remote.keys())[:30]:
        print(f"  {name}")
else:
    for name in ant_datasets:
        info = remote[name]
        print(f"  {name}")
        if hasattr(info, 'description'):
            print(f"    {info.description[:80]}")
        if hasattr(info, 'total_episodes'):
            print(f"    episodes: {info.total_episodes}")

## Cell 4 — Dataset 다운로드

가장 적합한 dataset 다운로드. Multi-freq 확보 가능성을 우선해서 *medium-replay* 또는 *medium-expert* 우선.

In [ ]:
# Cell 3에서 본 정확한 이름으로 수정 필요할 수 있음
# 후보 (최신 minari 명명규칙):
candidates = [
    'mujoco/ant/medium-replay-v0',
    'mujoco/ant/medium-replay-v1',
    'mujoco/ant/medium-expert-v0',
    'mujoco/ant/expert-v0',
    'D4RL/ant/medium-replay-v2',
    'D4RL/ant/medium-replay-v0',
]

dataset = None
used_name = None
for name in candidates:
    try:
        print(f"시도: {name}")
        dataset = minari.load_dataset(name, download=True)
        used_name = name
        print(f"  ✓ 성공!")
        break
    except Exception as e:
        print(f"  ✗ 실패: {str(e)[:80]}")

if dataset is None:
    print("\n⚠ 자동 후보 모두 실패. Cell 3의 출력을 보고 정확한 이름으로 수동 시도 필요.")
    print("예시:")
    print("  dataset = minari.load_dataset('정확한 이름', download=True)")
else:
    print(f"\n=== Loaded: {used_name} ===")
    print(f"Total episodes: {dataset.total_episodes}")
    print(f"Total steps: {dataset.total_steps}")
    print(f"Observation space: {dataset.observation_space}")
    print(f"Action space: {dataset.action_space}")

## Cell 5 — Dataset 구조 확인

Minari는 episodes generator를 제공. 한 episode를 가져와 shape 확인.

In [ ]:
# Sample episode 가져와서 구조 확인
episodes = list(dataset.iterate_episodes())
print(f"Total episodes loaded: {len(episodes)}")

ep0 = episodes[0]
print(f"\n=== Sample episode ===")
print(f"observations shape: {ep0.observations.shape}")
print(f"actions shape: {ep0.actions.shape}")
print(f"rewards shape: {ep0.rewards.shape}")
print(f"length: {len(ep0.actions)}")

# 우리가 원하는 obs 차원 (Ant-v5 = 105) vs dataset의 실제 차원
expected_obs_dim = 105
actual_obs_dim = ep0.observations.shape[-1]
print(f"\nExpected obs dim: {expected_obs_dim}")
print(f"Actual obs dim: {actual_obs_dim}")

if actual_obs_dim == expected_obs_dim:
    print("✓ Ant-v5와 호환")
elif actual_obs_dim < expected_obs_dim:
    print(f"⚠ Dataset이 더 적은 차원. {actual_obs_dim}만 사용.")
elif actual_obs_dim > expected_obs_dim:
    print(f"⚠ Dataset이 더 많은 차원. 첫 {expected_obs_dim}만 사용.")

# Episode length 분포
lengths = [len(ep.actions) for ep in episodes[:100]]
print(f"\nEpisode length 분포 (sample 100):")
print(f"  min={min(lengths)}, mean={np.mean(lengths):.0f}, max={max(lengths)}")

## Cell 6 — Phase 추출 + Quality 측정 함수

Plan B와 동일한 함수들.

In [ ]:
def extract_phase(joint_signal, smooth_sigma=2.0):
    """Hilbert transform으로 1D 신호의 순간 위상 추출."""
    s = joint_signal - joint_signal.mean()
    if smooth_sigma > 0:
        s = gaussian_filter1d(s, sigma=smooth_sigma)
    analytic = hilbert(s)
    phases = np.angle(analytic)
    phases = (phases + 2 * np.pi) % (2 * np.pi)
    return phases.astype(np.float32)


def measure_phase_quality(joint_signal, phases):
    """Episode의 phase 품질 측정."""
    L = len(phases)

    # Monotonicity
    phases_unwrapped = np.unwrap(phases)
    diffs = np.diff(phases_unwrapped)
    monotonicity = float((diffs > 0).mean())

    # FFT peak sharpness + estimated frequency
    signal_centered = joint_signal - joint_signal.mean()
    fft = np.abs(np.fft.rfft(signal_centered))
    if len(fft) > 1:
        peak_val = fft[1:].max()
        mean_val = fft[1:].mean()
        peak_sharpness = float(peak_val / (mean_val + 1e-8))
        peak_idx = fft[1:].argmax() + 1
        dt = 0.05
        estimated_freq = float(peak_idx / (L * dt))
    else:
        peak_sharpness = 0.0
        estimated_freq = 0.0

    # Frequency stability
    if len(diffs) > 0:
        inst_freq = diffs / (2 * np.pi * 0.05)
        freq_stability = float(np.std(inst_freq))
    else:
        freq_stability = float('inf')

    return {
        'monotonicity': monotonicity,
        'peak_sharpness': peak_sharpness,
        'estimated_freq': estimated_freq,
        'freq_stability': freq_stability,
    }


print("Phase 추출 함수 정의 완료")

## Cell 7 — 샘플 episode로 phase 추출 테스트

여러 hip joint candidate를 비교해서 가장 좋은 신호 선택.

In [ ]:
# Sample episode (충분히 긴 것)
sample_eps = [ep for ep in episodes[:20] if len(ep.actions) >= 200][:3]
print(f"Sample episodes (length >= 200): {len(sample_eps)}")

# obs[13:21]이 일반적으로 Ant joint angles
# 4개 hip의 인덱스 후보
hip_candidates = [13, 15, 17, 19]

print("\n=== 각 sample episode + 각 hip joint별 quality ===")
quality_summary = {idx: [] for idx in hip_candidates}

for ep_i, ep in enumerate(sample_eps):
    obs_seq = np.asarray(ep.observations)
    # 길이 맞추기 (last obs는 next state라 actions보다 1 더 김)
    obs_seq = obs_seq[:len(ep.actions)]

    print(f"\nEpisode {ep_i} (length {len(obs_seq)}):")
    for joint_idx in hip_candidates:
        if joint_idx >= obs_seq.shape[1]:
            continue
        sig = obs_seq[:, joint_idx]
        ph = extract_phase(sig)
        q = measure_phase_quality(sig, ph)
        quality_summary[joint_idx].append(q)
        print(f"  joint {joint_idx}: mono={q['monotonicity']:.2f}, "
              f"sharp={q['peak_sharpness']:.2f}, "
              f"freq={q['estimated_freq']:.2f}Hz, "
              f"stab={q['freq_stability']:.2f}")

# Joint별 평균 quality
print("\n=== Hip joint별 평균 quality (3 sample eps) ===")
for joint_idx, qs in quality_summary.items():
    if not qs:
        continue
    mean_mono = np.mean([q['monotonicity'] for q in qs])
    mean_sharp = np.mean([q['peak_sharpness'] for q in qs])
    mean_stab = np.mean([q['freq_stability'] for q in qs])
    print(f"  joint {joint_idx}: mono={mean_mono:.2f}, "
          f"sharp={mean_sharp:.2f}, stab={mean_stab:.2f}")

print("\n→ 가장 깔끔한 joint를 PHASE_JOINT_IDX로 선택")

## Cell 8 — Phase 추출용 Joint 선택

In [ ]:
# Cell 7 결과 보고 결정. 기본값 13.
PHASE_JOINT_IDX = 19   # 필요시 15, 17, 19 변경

print(f"Phase 라벨링용 joint: obs[{PHASE_JOINT_IDX}]")

## Cell 9 — Demo 추출 + Phase 라벨링 + Quality 필터링

전체 dataset (수만~수십만 episodes)에서 phase-coherent한 subset 자동 선택.

Plan B의 strict filter + frequency 균형을 dataset 전체에 적용. *통과율이 5%만 돼도 수천 episode 확보 가능*.

In [ ]:
def extract_demos_from_dataset(episodes, episode_length=200,
                                 min_episode_length=200,
                                 phase_joint_idx=13,
                                 expected_obs_dim=105,
                                 # Quality 임계값
                                 min_monotonicity=0.90,
                                 min_peak_sharpness=4.0,
                                 max_freq_stability=0.65,
                                 # Demo 개수 (single-freq reframe)
                                 max_demos=120,
                                 # 최대 처리 episode (시간 절약)
                                 max_episodes_to_process=5000):
    """Dataset의 episodes에서 phase-coherent demo 추출 (single-freq 버전).

    이전 버전과의 차이:
    - freq bin 균형 제거 → quality 기준 top-N 선택
    - max_demos 파라미터 추가
    - 학습 frequency window 통계를 metadata로 저장 (평가 시 in-dist 경계로 사용)
    """

    print(f"Phase 1: episodes 처리 (max {max_episodes_to_process})")
    enriched = []
    n_processed = 0
    n_too_short = 0

    for ep_i, ep in enumerate(episodes):
        if ep_i >= max_episodes_to_process:
            break

        L = min(len(ep.actions), episode_length)
        if L < min_episode_length:
            n_too_short += 1
            continue

        obs_seq = np.asarray(ep.observations)[:L, :expected_obs_dim]
        act_seq = np.asarray(ep.actions)[:L]

        joint_signal = obs_seq[:, phase_joint_idx]
        phases = extract_phase(joint_signal)
        quality = measure_phase_quality(joint_signal, phases)

        enriched.append({
            'obs': obs_seq,
            'act': act_seq,
            'phases': phases,
            'length': L,
            'reward_sum': float(np.sum(ep.rewards[:L])),
            'quality': quality,
        })
        n_processed += 1

        if (n_processed + n_too_short) % 500 == 0:
            print(f"  처리됨: {n_processed} (too short: {n_too_short})")

    print(f"\n총 처리: {n_processed} episodes (too short: {n_too_short})")

    if n_processed == 0:
        return None

    # Phase 2: Quality 분포 (전체 처리된 episodes)
    monos  = [e['quality']['monotonicity']   for e in enriched]
    sharps = [e['quality']['peak_sharpness'] for e in enriched]
    stabs  = [e['quality']['freq_stability'] for e in enriched]
    freqs  = [e['quality']['estimated_freq'] for e in enriched]
    print(f"\nPhase 2: Quality 분포 (전체 {n_processed} eps)")
    print(f"  mono:      min={min(monos):.2f}, mean={np.mean(monos):.2f}, max={max(monos):.2f}")
    print(f"  sharp:     min={min(sharps):.2f}, mean={np.mean(sharps):.2f}, max={max(sharps):.2f}")
    print(f"  freq_stab: min={min(stabs):.2f}, mean={np.mean(stabs):.2f}, max={max(stabs):.2f}")
    print(f"  freq:      min={min(freqs):.2f}, mean={np.mean(freqs):.2f}, max={max(freqs):.2f}")

    # 공통 quality score (filter 통과 후 ranking, fallback 양쪽에서 사용)
    def quality_score(e):
        return (e['quality']['peak_sharpness']
                + 5.0 * e['quality']['monotonicity']
                - e['quality']['freq_stability'])

    # Phase 3: 주기성 필터
    print(f"\nPhase 3: 주기성 필터")
    print(f"  mono >= {min_monotonicity}, sharp >= {min_peak_sharpness}, "
          f"stab <= {max_freq_stability}")
    passed = [e for e in enriched
              if e['quality']['monotonicity']   >= min_monotonicity
              and e['quality']['peak_sharpness'] >= min_peak_sharpness
              and e['quality']['freq_stability'] <= max_freq_stability]
    print(f"  통과: {len(passed)}/{len(enriched)} ({len(passed)/len(enriched)*100:.1f}%)")

    if len(passed) == 0:
        print("  → 통과 0개. 상위 10% fallback 적용.")
        scored = sorted(enriched, key=quality_score, reverse=True)
        passed = scored[:max(20, len(scored)//10)]
        print(f"  → {len(passed)} 선택 (fallback)")

    # Phase 4 (수정): Quality 기준 top-N (single-freq, bin 균형 제거)
    print(f"\nPhase 4: Quality 기준 top-{max_demos} 선택")
    passed.sort(key=quality_score, reverse=True)
    selected = passed[:max_demos]

    if len(selected) == 0:
        print("\n⚠ 선택 0개. 임계값 완화 필요.")
        return None

    # 학습 frequency window 통계 (평가 단계의 in-dist 경계 정의에 직접 사용)
    sel_freqs  = np.array([e['quality']['estimated_freq'] for e in selected])
    sel_monos  = np.array([e['quality']['monotonicity']   for e in selected])
    sel_sharps = np.array([e['quality']['peak_sharpness'] for e in selected])

    f_mean, f_std = float(sel_freqs.mean()), float(sel_freqs.std())
    f_min,  f_max = float(sel_freqs.min()),  float(sel_freqs.max())

    print(f"\n=== 선택된 {len(selected)} demos 특성 ===")
    print(f"  freq:  mean={f_mean:.3f}, std={f_std:.3f}, range=[{f_min:.3f}, {f_max:.3f}] Hz")
    print(f"  mono:  mean={sel_monos.mean():.3f}, min={sel_monos.min():.3f}")
    print(f"  sharp: mean={sel_sharps.mean():.3f}, min={sel_sharps.min():.3f}")
    print(f"\n  In-distribution window (mean±2σ): "
          f"[{f_mean - 2*f_std:.3f}, {f_mean + 2*f_std:.3f}] Hz")
    print(f"  → 학습된 frequency 영역. 평가 시 in-dist / mild OOD / future-work 경계로 사용.")

    if len(selected) < 50:
        print(f"\n⚠ 선택된 demos {len(selected)}개 < 50. "
              f"임계값 완화 또는 max_episodes_to_process 증가 권장.")

    # Numpy array로 변환
    n = len(selected)
    act_dim = selected[0]['act'].shape[-1]
    out = {
        'observations':    np.zeros((n, episode_length, expected_obs_dim), dtype=np.float32),
        'actions':         np.zeros((n, episode_length, act_dim),          dtype=np.float32),
        'phases':          np.zeros((n, episode_length),                   dtype=np.float32),
        'episode_lengths': np.zeros((n,),                                  dtype=np.int32),
        'estimated_freqs': np.zeros((n,),                                  dtype=np.float32),
        'monotonicity':    np.zeros((n,),                                  dtype=np.float32),
        'peak_sharpness':  np.zeros((n,),                                  dtype=np.float32),
        'freq_stability':  np.zeros((n,),                                  dtype=np.float32),
        # 학습 frequency window metadata
        'freq_window_mean': np.float32(f_mean),
        'freq_window_std':  np.float32(f_std),
        'freq_window_min':  np.float32(f_min),
        'freq_window_max':  np.float32(f_max),
    }
    for i, e in enumerate(selected):
        L = e['length']
        out['observations'][i, :L] = e['obs'][:L]
        out['actions'][i, :L]      = e['act'][:L]
        out['phases'][i, :L]       = e['phases'][:L]
        out['episode_lengths'][i]  = L
        out['estimated_freqs'][i]  = e['quality']['estimated_freq']
        out['monotonicity'][i]     = e['quality']['monotonicity']
        out['peak_sharpness'][i]   = e['quality']['peak_sharpness']
        out['freq_stability'][i]   = e['quality']['freq_stability']
    return out


# 실행
demos = extract_demos_from_dataset(
    episodes,
    episode_length=200,
    min_episode_length=200,
    phase_joint_idx=PHASE_JOINT_IDX,
    expected_obs_dim=105,
    min_monotonicity=0.90,
    min_peak_sharpness=8.0,
    max_freq_stability=0.5,
    max_demos=120,
    max_episodes_to_process=5000,
)

if demos is not None:
    np.savez_compressed('demos_ant_planC.npz', **demos)
    print(f"\n✓ 저장: demos_ant_planC.npz ({len(demos['observations'])} episodes)")
    print(f"  학습 freq window: "
          f"{float(demos['freq_window_mean']):.3f} ± {float(demos['freq_window_std']):.3f} Hz "
          f"(range [{float(demos['freq_window_min']):.3f}, {float(demos['freq_window_max']):.3f}])")

## Cell 10 — Drive 백업 ⭐

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ModuleNotFoundError:
    pass

import os
import shutil
import sys
from pathlib import Path

REPO_ROOT_CANDIDATES = [Path.cwd(), Path.cwd().parent]
REPO_ROOT = next((p for p in REPO_ROOT_CANDIDATES if (p / 'src' / 'paths.py').exists()), None)
assert REPO_ROOT is not None, 'repo root with src/paths.py not found; run this notebook from the cloned repository'
SRC_DIR = REPO_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from paths import ARTIFACT_ROOT, DATA_DIR, FIGURES_DIR, ensure_artifact_dirs
ensure_artifact_dirs()

if os.path.exists('demos_ant_planC.npz'):
    dest = DATA_DIR / 'demos_ant_planC.npz'
    shutil.copy('demos_ant_planC.npz', dest)
    size_mb = dest.stat().st_size / 1e6
    print(f'  ✓ {dest} ({size_mb:.1f} MB)')

print(f'\nArtifact data dir: {DATA_DIR}')
print(f'Files: {[p.name for p in DATA_DIR.iterdir()]}')


## Cell 11 — Demo 품질 검증 시각화

In [ ]:
demos = np.load(DATA_DIR / 'demos_ant_planC.npz')
print(f"Loaded: {list(demos.keys())}")
print(f"Episodes: {len(demos['observations'])}")
print(f"평균 length: {demos['episode_lengths'].mean():.1f}")

# Quality 분포 시각화
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, key, color in zip(
    axes,
    ['monotonicity', 'peak_sharpness', 'freq_stability', 'estimated_freqs'],
    ['green', 'blue', 'orange', 'purple']):
    ax.hist(demos[key], bins=20, edgecolor='black', color=color, alpha=0.7)
    ax.axvline(demos[key].mean(), color='red', linestyle='--', label=f"mean {demos[key].mean():.2f}")
    ax.set_xlabel(key)
    ax.set_title(key)
    ax.legend()
    ax.grid(True, alpha=0.3)
plt.tight_layout()
out_png = FIGURES_DIR / 'planC_quality_dist.png'
plt.savefig(out_png, dpi=80)
print(f'✓ 저장: {out_png}')
plt.show()
plt.close('all')
gc.collect()

# 3개 episode (다른 freq) 시각화
sorted_idx = np.argsort(demos['estimated_freqs'])
sample_indices = [sorted_idx[0], sorted_idx[len(sorted_idx)//2], sorted_idx[-1]]

fig, axes = plt.subplots(3, 3, figsize=(14, 8))
for row, ep_idx in enumerate(sample_indices):
    L = demos['episode_lengths'][ep_idx]
    f = demos['estimated_freqs'][ep_idx]

    axes[row, 0].plot(demos['observations'][ep_idx, :L, PHASE_JOINT_IDX])
    axes[row, 0].set_title(f"Ep{ep_idx}: joint signal (freq={f:.2f}Hz)")
    axes[row, 0].grid(True)

    axes[row, 1].plot(demos['phases'][ep_idx, :L])
    axes[row, 1].set_title("Extracted phase")
    axes[row, 1].grid(True)

    axes[row, 2].scatter(demos['phases'][ep_idx, :L],
                         demos['actions'][ep_idx, :L, 0],
                         s=4, alpha=0.5)
    axes[row, 2].set_xlabel('Phase')
    axes[row, 2].set_ylabel('Action[0]')
    axes[row, 2].set_title('Phase ↔ Action')
    axes[row, 2].grid(True)

plt.tight_layout()
out_png = FIGURES_DIR / 'planC_demo_visualization.png'
plt.savefig(out_png, dpi=80)
print(f'✓ 저장: {out_png}')
plt.show()
plt.close('all')
gc.collect()



## Cell 12 — 자동 품질 평가

In [ ]:
print("=== Plan C Demo 품질 자동 평가 ===\n")

n = len(demos['observations'])
print(f"1. Episode 수: {n}")
print(f"   {'✓ 충분 (50+)' if n >= 50 else '○ 적당 (30-50)' if n >= 30 else '⚠ 부족 (<30)'}")

mean_len = demos['episode_lengths'].mean()
print(f"\n2. 평균 length: {mean_len:.1f}")
print(f"   {'✓ 우수' if mean_len >= 200 else '○ 충분' if mean_len >= 150 else '⚠ 짧음'}")

freq_std = demos['estimated_freqs'].std()
freq_range = demos['estimated_freqs'].max() - demos['estimated_freqs'].min()
print(f"\n3. Frequency 다양성: std={freq_std:.3f}, range={freq_range:.3f}")
print(f"   {'✓ 매우 다양' if freq_std > 0.3 else '○ 적당' if freq_std > 0.1 else '⚠ 단조'}")

# Frequency bin별 분포
print(f"\n4. Frequency bin별 분포:")
for lo, hi in [(0.5, 1.5), (1.5, 2.5), (2.5, 3.5)]:
    cnt = ((demos['estimated_freqs'] >= lo) & (demos['estimated_freqs'] < hi)).sum()
    print(f"   freq [{lo:.1f}, {hi:.1f}): {cnt} eps {'✓' if cnt >= 5 else '⚠'}")

mean_mono = demos['monotonicity'].mean()
mean_sharp = demos['peak_sharpness'].mean()
print(f"\n5. 평균 phase quality: mono={mean_mono:.2f}, sharp={mean_sharp:.2f}")
print(f"   {'✓ 매우 깔끔' if mean_mono > 0.95 and mean_sharp > 5 else '○ 충분'}")

print("\n=== 종합 ===")
print("Plan C 결과가 Plan B (17 ep, 단일 freq)보다 좋으면 → 다음 단계 (diffusion policy) 진행")
print("비슷하거나 안 좋으면 → Plan B 결과 사용 또는 다른 dataset 시도")

## Cell 13 — Plan B와 비교 (선택)

Plan B의 결과 파일 (`demos_ant_planB_strict.npz`)이 있으면 비교.

In [ ]:
planB_path = 'demos_ant_planB_strict.npz'
planC_path = 'demos_ant_planC.npz'

if os.path.exists(planB_path):
    demos_B = np.load(planB_path)
    print("=== Plan B vs Plan C 비교 ===")
    print(f"{'':25} {'Plan B':>12} {'Plan C':>12}")
    print(f"{'Episodes':25} {len(demos_B['observations']):>12} {len(demos['observations']):>12}")
    print(f"{'평균 length':25} {demos_B['episode_lengths'].mean():>12.1f} {demos['episode_lengths'].mean():>12.1f}")
    print(f"{'평균 mono':25} {demos_B['monotonicity'].mean():>12.2f} {demos['monotonicity'].mean():>12.2f}")
    print(f"{'평균 sharp':25} {demos_B['peak_sharpness'].mean():>12.2f} {demos['peak_sharpness'].mean():>12.2f}")
    print(f"{'평균 freq':25} {demos_B['estimated_freqs'].mean():>12.2f} {demos['estimated_freqs'].mean():>12.2f}")
    print(f"{'freq std':25} {demos_B['estimated_freqs'].std():>12.3f} {demos['estimated_freqs'].std():>12.3f}")
else:
    print("Plan B 파일 없음. 비교 생략.")

## 완료 후

산출물:
- [ ] `demos_ant_planC.npz` (D4RL/Minari 기반 demo)
- [ ] Drive 백업
- [ ] 품질 metric: episodes 50+, freq 다양성, mono 0.9+, sharp 4+

다음 노트북: **Phase-Conditioned Diffusion Policy** 모델 구현.

## 주의사항

**Cell 4에서 dataset 다운로드 실패 시**:
- Cell 3 출력의 *정확한* dataset 이름으로 수동 시도
- Minari가 안 되면 D4RL 직접 시도:
  ```python
  !pip install d4rl --quiet
  import gym
  env = gym.make('ant-medium-replay-v2')
  data = env.get_dataset()
  ```

**Cell 5에서 obs dim 불일치 시**:
- Dataset이 obs 27차원 (legacy Ant) 또는 111차원 (v5 with contact)일 수 있음
- 27 → 105 변환 까다로움. v5 (105) 또는 v4 (27) 중 하나로 통일 권장.
- 만약 27차원이면 우리 framing 전체를 v4로 전환하거나, 다른 dataset 시도.

**무엇이 Plan B보다 좋아야 하는가**:
- Episode 수: 17 → 50+
- Frequency 다양성: 단일 → 여러 bin
- 만약 둘 다 비슷하면 Plan C의 의미 없음 → Plan B 결과 사용
